# Needle MASTER report and reproducibility notebook

Local draft; cloud execution not verified. Fill team details and a private downloadable code link before submission. Never publish Needle homework solutions publicly.

# Memory-Efficient MASTER in Needle: Exact Cross-Stock Attention with Recomputation

Local report draft, 26 October 2026. All reported experiments ran on the local
Windows workstation. The accompanying notebook has not yet been executed in
Google Colab. Team information and the final private code-download link remain
to be supplied before course submission.



## Motivation and scope

Cross-stock attention offers a useful systems workload because its sequence axis
is the number of stocks observed together. A forecasting model may use only a
short temporal history yet attend over hundreds or thousands of securities at
each historical step. Materializing every pairwise probability can therefore
consume more memory than the projected features themselves. Our project asks
whether Needle can execute this workload while retaining substantially less
attention state, and what additional computation that saving requires.

We implement a small MASTER-style architecture and a new CUDA execution path for
Needle. The architecture provides a concrete application rather than a claim of
new forecasting methodology. The systems contribution combines device storage,
custom attention forward and backward kernels, an explicit first-order gradient
contract, numerical checks, and complete-model measurements. We intentionally
separate implementation correctness, resource efficiency, and forecasting
behavior. A model that fits a small dataset does not by itself establish either
kernel correctness or an economically useful strategy.



## Background and attribution

Li and colleagues introduced MASTER, a market-guided stock transformer accepted
at AAAI 2024. Its market gate and alternating intra-stock/inter-stock aggregation
motivate the application architecture. We follow the main structure in an existing
local research implementation, while using a compact daily-input variant instead
of claiming reproduction of the paper's complete data pipeline or published
scores. We cite the architecture and distinguish earlier work from this project.

Dao and colleagues' FlashAttention work motivates stable online normalization,
tiling and probability recomputation. Those algorithmic ideas are established
background, not our invention. Our kernels do not implement the complete
FlashAttention scheduling or tensor-core design. The course contribution is a
Needle implementation and its empirical study. The completed local HW2 Needle
library is retained as an unchanged, hashed dependency; its source files are not
counted as new project work. NumPy, CuPy and cuBLAS are external numerical
dependencies, and PyTorch appears only in independent reference checks.



## Framework integration

The CPU feasibility version already used Needle TensorOps and modules, but its
operators ran on NumPy arrays. The GPU version adds a process-wide CUDA array
backend. Tensor construction uploads arrays, Tensor storage remains on the device,
and the basic mathematical and logarithmic operators dispatch to CuPy. The graph
and model interfaces remain Needle's. Basic matrix multiplication uses cuBLAS;
the attention-specific CUDA code is compiled through NVRTC and launched on the
current stream. Training never calls PyTorch autograd or attention routines.

This adapter is a supported path for this model, not a complete replacement for
every Needle backend. It does not load the Linux HW3 extension into Windows or
claim that all HW3 low-level operators have been independently reimplemented.
CPU and CUDA tests run in separate processes, and mixed-device graphs are not
supported. CUDA dependencies are isolated from the existing quantitative research
environment. On Windows, compiler-facing dependency paths use an ASCII junction
when the actual project path contains Chinese characters; the underlying files
remain in the requested project directory.

The model includes a market-status gate, a feature projection, sinusoidal
positions, temporal attention, cross-stock attention, temporal pooling and a
scalar prediction head. Both attention blocks include normalization and a small
feed-forward residual component. Temporal attention is unscaled; stock attention
uses inverse square root of head dimension, matching the local design reference.
All temporal inputs are historical at the signal date. Dropout is fixed to zero,
and this experiment does not require causal or padding masks.



## Tiled forward and backward

Dense attention computes a scaled QK transpose, normalizes each row, then
multiplies by V. Saving the probability matrix costs quadratic storage in the
number of stocks. Our forward kernel assigns one block to a query row and leading
batch. Threads compute scores for a tile of keys, reduce a maximum, exponentiate
relative scores, and accumulate weighted values. When a later tile has a larger
maximum, the earlier numerator and denominator are rescaled before accumulation.
This online softmax produces the same mathematical operation without a global
quadratic probability tensor.

The saved state contains Q, K, V, output O, and row log-normalizers. During
backward, probabilities are reconstructed for the required tile. The softmax
adjoint uses the row correction given by the inner product of dO and O. One
kernel owns each query row and accumulates dQ; another owns each key row and
accumulates dK and dV. This ownership avoids floating-point atomics, but repeats
some probability calculations across the two backward passes.

For fixed head dimension and batch size, saved attention state grows linearly
with stock count. Arithmetic remains quadratic. The implementation uses scalar
dot products and shared reductions rather than sophisticated matrix tiles.
Consequently, memory savings can come with worse latency. Supported FP32 block
sizes are 32, 64, 128 and 256, with feature dimensions no larger than the block.
Other GPU shapes and FP64 use a separate batched CuPy tiled reference.



## Gradient lifetime and correctness

Complete-model testing exposed a framework issue that attention-only testing did
not reveal. The HW2 teaching driver assigned gradients to constant buffers and
constructed a differentiable backward graph. Persistent position tensors could
retain earlier graphs. The CUDA backend therefore uses an explicit first-order
driver: shared-input contributions accumulate, each VJP is detached, and constants
are skipped. All GPU comparison arms use this same driver. Higher-order gradients
are outside the interface. A regression runs twenty updates and verifies that
constant buffers have no gradients and that live allocations remain unchanged
between steps five and twenty. Pre-fix measurements are archived and excluded.

CPU verification covers finite differences, partial tiles, shared inputs,
FP32/FP64 results, stock permutation equivariance and short optimizer trajectories.
GPU kernel checks compare outputs and Q/K/V gradients with an independent
PyTorch CPU oracle across several shapes and block lengths. Full-model checks
use FP64 to reduce cancellation effects. A separate real-date test computes the
correlation loss and checks every parameter gradient against the reference.
The accompanying result files contain the actual maximum errors, not simply a
statement that tests passed.

Exact attention means mathematical equivalence, not identical floating-point
rounding. FP32 checks initially found relative discrepancies on gradient entries
near zero. We retained that observation, added normwise error measurements and
used an FP64 architectural oracle. We did not describe a relaxed FP32 comparison
as bitwise agreement. Training trajectories require a separate investigation
because repeated adaptive updates can amplify small arithmetic differences.



## Real-data protocol and timing safety

The integration panel uses 128 securities chosen by a deterministic code hash
from stocks already listed by 2013 and not delisted as of the initial 2014 signal
date. Later survival and returns do not enter selection. The source warehouse is
opened read-only. A manifest records SQL, source views, dates, features, units,
normalization parameters, exclusions and an array fingerprint. No network
download or production-table mutation occurs.

Inputs contain twelve daily observations of nine price/activity/size features
and a traded indicator. Missing historical values become zero after training-set
standardization; the indicator preserves the missing-trading distinction. Market
inputs use rolling statistics available through each signal date. Scaling is fit
only on 2014 observations. A prefix recomputation checks that a validation-date
market vector is unchanged when subsequent source observations are unavailable.
Historical provider corrections remain a data-vintage limitation.

Training uses 2014 signal dates, validation uses the first half of 2015, and test
uses the first half of 2016. Each segment loses its final five candidate dates so
its forward five-trading-day label is realized within the segment. The latest
training label precedes the earliest validation signal, and the same condition
holds between validation and test. A stock with a known tradable T-day observation
enters scoring even when its future label is missing. Only the loss and metrics
mask that missing target. The report includes counts of those scored-but-unlabeled
rows rather than silently removing them.

The fixed experiment uses width 32, seed 42, three epochs, Adam with learning
rate 0.001, and gradient norm clipping at one. There is no validation-driven
epoch selection and no use of post-2017 observations. This bounded daily variant
is a systems demonstration, not the earlier production monthly PCA model.



## Measurements and findings

GPU benchmarks use the local RTX 5080 Laptop GPU. Each configuration runs in a
fresh process, performs two warmups, and records five timed iterations. A model
step includes forward, scalar loss, backward and Adam. We record synchronized
wall time and CUDA-event time separately. An allocator wrapper captures peak
live CuPy-pool allocations after each allocation. This excludes the CUDA context
and external library workspaces; it is neither whole-device usage nor CPU RSS.

At 1,024 stocks and twelve time steps, the measured full-model live-pool peak
was about 1,461 MiB for ordinary dense graph attention, 397 MiB for dense attention
with a handwritten VJP, and 159 MiB for tiled recomputation. The handwritten dense
control is important: it prevents attributing every difference to tiling when
framework graph construction can also contribute. Relative to that control, the
tiled model used approximately sixty percent less measured live memory. Its
median wall step increased from about 15.5 to 21.6 milliseconds. These local
measurements support a storage/recomputation tradeoff, not a universal speedup.

Predefined ablations vary stock count, temporal length and block length. The
attention-only measurement is reported separately from the complete model.
The model's other activations limit the end-to-end saving, and host launch costs
can dominate small configurations. Five iterations on one Windows laptop are
insufficient for broad performance claims across architectures or power states.

All three implementations trained on the fixed real panel, with complete saved
weights, scores and date/code mappings. Their FP32 trajectories diverged over
hundreds of updates. A train-only sixteen-step diagnostic found exact same-kernel
replay, much smaller differences in FP64, and larger parameter differences in
FP32. This supports numerical sensitivity as an explanation in the tested setting,
but does not establish predictive equivalence after long training. We report the
single-seed RankIC observations without selecting the apparent winner or claiming
an attention method improves forecasting.



## Limitations and deliverables

The project supports first-order, fixed-length, dropout-free execution. Masks,
mixed-device graphs, higher-order derivatives and complete FlashAttention-style
optimization are not implemented. The real cohort is small and historically fixed.
Labels are adjusted close-to-close forecasts, with no execution model, costs,
price limits or ST constraints. Thus no backtest or investment conclusion follows.
The notebook includes reproducible commands and original figures, but cloud
execution remains to be verified. The final course submission also needs team
details, a private code link and the required presentation video.

The current evidence establishes a functioning Needle GPU extension, independently
checked derivatives, real-panel integration, and measured full-model resource
tradeoffs. Future work should target kernel scheduling and memory coalescing,
stronger FP32 stability, additional devices and repeated independent performance
runs. Those extensions should be evaluated on their own evidence rather than
assumed from the memory reductions already observed.



## References

- Tong Li et al., MASTER: Market-Guided Stock Transformer for Stock Price Forecasting,
  AAAI 2024. https://doi.org/10.1609/aaai.v38i1.27767
- Tri Dao et al., FlashAttention: Fast and Memory-Efficient Exact Attention with
  IO-Awareness, 2022. https://arxiv.org/abs/2205.14135
- Deep Learning Systems final project requirements. https://dlsyscourse.org/project/
- CuPy installation and dependency documentation. https://docs.cupy.dev/en/stable/install.html


## Measured results

# GPU 工程结果与复现记录

日期：2026-10-26。当前项目位置：`C:\学习资料\研2\DEEP LEARNING SYSTEM\final_projects\needle_master`。

**已完成 CUDA 后端、自写 attention 内核、真实小面板训练及完整模型实验。以下是工程证据，不代表最终课程提交或投资结果。**

## GPU 完整模型基准

RTX 5080 Laptop GPU，FP32，width=32，T=12；每配置独立子进程，2 次 warmup、5 次测量，中位数。训练步含前向、loss、反向与 Adam。

| 股票数 | attention 实现 | 内存池存活峰值 MiB | 训练步 wall ms |
|---:|---|---:|---:|
| 128 | dense | 37.5 | 14.53 |
| 128 | dense_vjp | 21.8 | 14.13 |
| 128 | tiled | 20.0 | 13.72 |
| 512 | dense | 394.5 | 14.47 |
| 512 | dense_vjp | 126.6 | 13.94 |
| 512 | tiled | 79.5 | 14.61 |
| 1024 | dense | 1460.9 | 16.55 |
| 1024 | dense_vjp | 397.0 | 15.66 |
| 1024 | tiled | 158.7 | 21.81 |

1024 只股票时，相对稠密手写 VJP 对照，分块版降低约 **60.0%** 的内存池峰值，训练步耗时增加约 **39.2%**。
内存口径是 CuPy allocator 的存活分配峰值，排除 CUDA context 和 allocator 之外的库工作区；不是整卡显存、缓存保留量或 CPU RSS。
所有 GPU 实现使用同一一阶驱动，避免把不必要的反向图保留算作分块收益。修复前、迁移前基准均已另存，不混入当前结论。

## 股票数、时间长度、块长消融

| 类型 | 实现 | N | T | 块长 | 峰值 MiB | wall ms |
|---|---|---:|---:|---:|---:|---:|
| model | tiled | 512 | 12 | 32 | 79.5 | 13.83 |
| model | tiled | 512 | 12 | 128 | 79.5 | 14.41 |
| model | dense_vjp | 512 | 4 | 64 | 42.2 | 14.73 |
| model | tiled | 512 | 4 | 64 | 27.7 | 13.52 |
| model | dense_vjp | 512 | 24 | 64 | 255.1 | 14.74 |
| model | tiled | 512 | 24 | 64 | 158.5 | 16.69 |
| attention | dense_vjp | 1024 | 12 | 64 | 300.0 | 4.60 |
| attention | tiled | 1024 | 12 | 64 | 12.2 | 12.97 |

## 正确性和内存生命周期

- CPU 9 项原有检查通过；GPU 9 个 FP32 attention 形状/块长组合通过，最大输出误差 2.980e-07，最大梯度误差 7.153e-07。
- 完整模型 FP64 三种实现与独立 PyTorch CPU 对照；真实日期 20140102 的全参数梯度最大误差 4.937e-13。
- FP32 全模型单步逐参数 relative-L2 最大误差 1.547e-04；近零元素的相对误差不能当作严格逐位一致。
- 修复 HW2 驱动对常量保存梯度的问题。一阶 VJP 分离、跳过常量；第 5/20 步常量梯度与内存不增长回归通过。

## 真实小面板：数据与 PIT 边界

固定历史 cohort 为 128 只，选择依据为 2014-01-02 已知上市/退市状态和代码哈希，不使用后来的收益或是否存活。
12 个日级时间步、10 个输入特征；不是原生产月度 PCA 模型。源库只读，没有下载或改表。

| 段 | 信号日期 | 标签最晚实现日 | 信号日数 | 打分股票日 | 缺标签但仍打分 |
|---|---|---|---:|---:|---:|
| train | 20140102–20141224 | 20141231 | 240 | 28178 | 416 |
| validation | 20150105–20150623 | 20150630 | 114 | 12787 | 357 |
| test | 20160104–20160623 | 20160630 | 115 | 13465 | 208 |

各段末尾剔除 5 个标签跨界信号日。训练标签最晚 2014-12-31，验证最早信号 2015-01-05；验证标签最晚 2015-06-30，测试最早信号 2016-01-04。
特征与市场输入归一化只拟合 2014 年。2015-03-02 市场特征通过历史前缀重算；数组指纹与打分/标签统计审计通过。
标签为 t 到 t+5 的复权收盘对数收益；未来标签缺失只屏蔽 loss/指标。历史源库可能包含供应商修订，不能声称拥有当年的原始数据 vintage。

## 固定 3 epoch 训练结果

seed=42、width=32、Adam lr=0.001、clip=1、dropout=0；没有按验证/测试选择 epoch 或调参。

| 实现 | train loss 1→3 | 验证 mean daily RankIC | 测试 mean daily RankIC |
|---|---|---:|---:|
| dense | 0.9162→0.8243 | -0.0120 | 0.0459 |
| dense_vjp | 0.9406→0.8590 | -0.0076 | 0.0366 |
| tiled | 0.9449→0.8647 | 0.0625 | 0.0432 |

**三种 FP32 长训练轨迹没有保持一致，不能据此选出预测更好的 attention 实现。**
仅用训练日的 16 步诊断：FP32 参数最大差 1.269e-03，FP64 参数最大差 4.754e-13；同一 CUDA 实现重放参数差为 0。
结果支持本测试中舍入误差与自适应更新的敏感性解释；不等于已证明任意长期训练的预测等价。测试集结果未用于改选配置。
权重、分数和日期/股票映射保存在 `results/real_training_v1/`。原始完整训练在迁移前完成；数据和源码随项目保留，迁移后关键检查与 GPU 基准已重跑。

## 当前边界

- 新增的是 Needle CUDA/CuPy 数组后端，未直接使用 HW3 Linux NDArray 扩展；CuPy/cuBLAS 提供基础运算，attention 前向/反向是本项目 CUDA 代码。
- 不支持 mixed-device、二阶梯度、attention mask 或训练 dropout。固定长度、dropout=0 的当前模型适用。
- GPU FP32 内核支持的块长/维度有限；FP64 走另一条批量分块参考实现。
- 小型历史股票 cohort、单 seed；没有交易成本、ST/涨跌停规则或可执行回测，不作收益判断。
- notebook 为本地草稿，尚未实跑 Google Colab，组员信息、私有代码链接和 90 秒视频仍需最终交付时补齐。

## 复现

`run_gpu_checks.ps1`、`scripts/benchmark_gpu.py`、`scripts/train_real_gpu.py`。
课程要求：[Final Project](https://dlsyscourse.org/project/)。算法背景：[MASTER](https://doi.org/10.1609/aaai.v38i1.27767)、[FlashAttention](https://arxiv.org/abs/2205.14135)。


## Reproduce with the private project bundle

Choose a GPU runtime for GPU checks. Upload `needle_master_project.zip` privately. The archive includes the small derived panel, scripts and local evidence, but no CUDA binaries or source warehouse. The following cells have not been executed in Colab.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded=files.upload()
archive=Path('needle_master_project.zip')
assert archive.exists()
root=Path('/content/needle_master').resolve()
root.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for item in z.infolist():
        target=(root/item.filename).resolve()
        assert target.is_relative_to(root)
    z.extractall(root)
os.chdir(root)
print(root)


In [ ]:
import subprocess,sys,importlib.util
subprocess.check_call([sys.executable,'-m','pip','install','-q','-r','requirements.txt'])
if importlib.util.find_spec('cupy') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','cupy-cuda12x[ctk]==14.2.0'])
subprocess.check_call([sys.executable,'scripts/verify.py'])


In [ ]:
import subprocess,sys
for script in ['scripts/verify_gpu.py','scripts/verify_real_alignment.py','scripts/check_training_numerics.py']:
    subprocess.check_call([sys.executable,script])


In [ ]:
# This reruns measurements on the current GPU; results may differ from the laptop.
subprocess.check_call([sys.executable,'scripts/benchmark_gpu.py'])
# Optional bounded real training; creates a new versioned run and keeps old evidence.
# subprocess.check_call([sys.executable,'scripts/train_real_gpu.py'])


In [ ]:
from IPython.display import display,Image
display(Image(filename='figures/gpu_tradeoff.png'))
